In [14]:
# ==============================================================================
# SCRIPT:         colab_vektor_engine_v5.4_validation_col.ipynb
# OPERATION:      Operation: Vektor v5.4 (Validation Column Protocol)
# PURPOSE:        The definitive asset. Appends the 'session_id' used for
#                 processing to the final output file to allow for direct
#                 validation of the session logic.
# ==============================================================================

# --- 1. SETUP ---
!pip install -q gspread folium
import pandas as pd
import numpy as np
import os
import gspread
import folium
import time
from google.colab import auth, drive
from google.auth import default

print("Authenticating & Mounting...")
auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)
drive.mount('/content/drive', force_remount=True)
print("✅ Setup complete.")

# --- 2. CONFIGURATION ---
ANCHORS_SPREADSHEET_NAME, ANCHORS_WORKSHEET_NAME = 'GTS-4', '250918'
TARGETS_SPREADSHEET_NAME, TARGETS_WORKSHEET_NAME = 'raw_Offers', 'raw_requests_VVSpolished'
ANCHOR_TS_COL, ANCHOR_LAT_COL, ANCHOR_LON_COL = 'realTimestamp', 'latitude', 'longitude'
TARGET_TS_COL, TARGET_ID_COL = 'timestamp', 'Obs_ID'
TARGET_SESSION_COL = 'Session_ID' # Using ground-truth 'Session_ID'
OUTPUT_FOLDER_PATH = '/content/drive/MyDrive/_Pienza/Assets/Database/03_analysis'
OUTPUT_FILENAME = 'raw_Offers_with_Inferred_Vektor_v5.4_PRESERVED.csv'

# --- 3. BATTLE-TESTED UTILITY FUNCTIONS ---
def make_headers_unique(header_list):
    counts, unique_headers = {}, []
    for i, header in enumerate(header_list):
        header = str(header).strip() if str(header).strip() != "" else f"blank_{i}"
        if header not in counts:
            counts[header] = 1
            unique_headers.append(header)
        else:
            counts[header] += 1
            unique_headers.append(f"{header}_{counts[header]}")
    return unique_headers

def haversine_distance(lat1, lon1, lat2, lon2): R=6371000; phi1,phi2=np.radians(lat1),np.radians(lat2); dphi,dlambda=np.radians(lat2-lat1),np.radians(lon2-lon1); a=np.sin(dphi/2)**2+np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2; return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a))
def calculate_bearing(lat1, lon1, lat2, lon2): lat1,lon1,lat2,lon2=np.radians(lat1),np.radians(lon1),np.radians(lat2),np.radians(lon2); dLon=lon2-lon1; x=np.cos(lat2)*np.sin(dLon); y=np.cos(lat1)*np.sin(lat2)-np.sin(lat1)*np.cos(lat2)*np.cos(dLon); return (np.degrees(np.arctan2(x,y))+360)%360

try:
    # --- 4. STAGE 1: Data Ingestion & Ground-Truth Session Annotation ---
    print("\n--- STAGE 1: Data Ingestion & Session Annotation ---")

    target_sheet=gc.open(TARGETS_SPREADSHEET_NAME).worksheet(TARGETS_WORKSHEET_NAME)
    df_targets_master = pd.DataFrame(target_sheet.get_all_values()[1:], columns=make_headers_unique(target_sheet.get_all_values()[0]))
    df_targets_master.replace('', np.nan, inplace=True)
    initial_target_count = len(df_targets_master)
    df_targets_master.set_index(TARGET_ID_COL, inplace=True)
    print(f"  > Master target dataframe loaded with {initial_target_count} rows.")

    anchor_sheet=gc.open(ANCHORS_SPREADSHEET_NAME).worksheet(ANCHORS_WORKSHEET_NAME)
    df_anchors = pd.DataFrame(anchor_sheet.get_all_values()[1:], columns=make_headers_unique(anchor_sheet.get_all_values()[0]))
    df_anchors.replace('', np.nan, inplace=True)
    df_anchors['timestamp_utc']=pd.to_datetime(df_anchors[ANCHOR_TS_COL], errors='coerce')
    df_anchors.dropna(subset=['timestamp_utc'], inplace=True)
    df_anchors[ANCHOR_LAT_COL]=pd.to_numeric(df_anchors[ANCHOR_LAT_COL], errors='coerce')
    df_anchors[ANCHOR_LON_COL]=pd.to_numeric(df_anchors[ANCHOR_LON_COL], errors='coerce')
    df_anchors['timestamp_utc']=df_anchors['timestamp_utc'].dt.tz_localize('UTC')
    df_anchors.sort_values(by='timestamp_utc', inplace=True)

    print("  > Annotating anchors with ground-truth sessions...")
    session_map = df_targets_master.copy().reset_index()
    session_map['timestamp_utc'] = pd.to_datetime(session_map[TARGET_TS_COL], format='%Y:%m:%d %H:%M:%S', errors='coerce')
    session_map.dropna(subset=['timestamp_utc', TARGET_SESSION_COL], inplace=True)
    session_map['timestamp_utc'] = session_map['timestamp_utc'].dt.tz_localize('UTC')
    session_map = session_map[['timestamp_utc', TARGET_SESSION_COL]].sort_values('timestamp_utc')

    df_anchors = pd.merge_asof(df_anchors, session_map, on='timestamp_utc', direction='backward')
    df_anchors[TARGET_SESSION_COL].fillna(method='bfill', inplace=True)
    df_anchors.dropna(subset=[TARGET_SESSION_COL], inplace=True)
    print(f"  > Anchors successfully annotated.")

    df_targets_clean = df_targets_master.copy().reset_index()
    df_targets_clean['timestamp_utc'] = pd.to_datetime(df_targets_clean[TARGET_TS_COL], format='%Y:%m:%d %H:%M:%S', errors='coerce')
    df_targets_clean.dropna(subset=['timestamp_utc', TARGET_ID_COL, TARGET_SESSION_COL], inplace=True)
    df_targets_clean['timestamp_utc'] = df_targets_clean['timestamp_utc'].dt.tz_localize('UTC')
    df_targets_clean.sort_values(by='timestamp_utc', inplace=True)
    print(f"  > Cleaned dataframe for processing contains {len(df_targets_clean)} valid rows.")

    # --- 5. STAGE 2: Session-Aware Merge (Using Ground-Truth Key) ---
    print("\n--- STAGE 2: Definitive Session-Aware Merge ---")
    df_before = pd.merge_asof(df_targets_clean, df_anchors, on='timestamp_utc', by=TARGET_SESSION_COL, direction='backward')
    df_before.rename(columns={'latitude': 'latitude_before', 'longitude': 'longitude_before', 'realTimestamp':'timestamp_before'}, inplace=True)
    df_after = pd.merge_asof(df_targets_clean, df_anchors, on='timestamp_utc', by=TARGET_SESSION_COL, direction='forward')
    df_after.rename(columns={'latitude': 'latitude_after', 'longitude': 'longitude_after', 'realTimestamp':'timestamp_after'}, inplace=True)
    df_merged = df_before.join(df_after[['latitude_after', 'longitude_after', 'timestamp_after']])
    print("  > Explicit merge-rename-join protocol complete.")

    # --- 6. STAGE 3: Vektor Calculation ---
    print("\n--- STAGE 3: Calculating Inferred State Vektors ---")
    df_merged['timestamp_before_utc'] = pd.to_datetime(df_merged['timestamp_before'], errors='coerce').dt.tz_localize('UTC')
    df_merged['timestamp_after_utc'] = pd.to_datetime(df_merged['timestamp_after'], errors='coerce').dt.tz_localize('UTC')
    time_total = (df_merged['timestamp_after_utc'] - df_merged['timestamp_before_utc']).dt.total_seconds()
    time_partial = (df_merged['timestamp_utc'] - df_merged['timestamp_before_utc']).dt.total_seconds()
    ratio = (time_partial / time_total).clip(0, 1).fillna(0)
    distance = haversine_distance(df_merged['latitude_before'], df_merged['longitude_before'], df_merged['latitude_after'], df_merged['longitude_after'])
    df_merged['inferred_agent_speed_mps'] = np.divide(distance, time_total)
    df_merged['inferred_agent_bearing'] = calculate_bearing(df_merged['latitude_before'], df_merged['longitude_before'], df_merged['latitude_after'], df_merged['longitude_after'])
    df_merged['inferred_agent_lat'] = df_merged['latitude_before'] + (df_merged['latitude_after'] - df_merged['latitude_before']) * ratio
    df_merged['inferred_agent_lon'] = df_merged['longitude_before'] + (df_merged['longitude_after'] - df_merged['longitude_before']) * ratio
    print("  > Vektor calculations complete.")

    # --- 7. STAGE 4: ADVANCED Edge Case Handling ---
    print("\n--- STAGE 4: Advanced Edge Case Handling ---")
    unanchored = df_merged['latitude_before'].isna() & df_merged['latitude_after'].isna()
    start = df_merged['latitude_before'].isna() & ~df_merged['latitude_after'].isna()
    end = ~df_merged['latitude_before'].isna() & df_merged['latitude_after'].isna()
    stationary = (df_merged['latitude_before'] == df_merged['latitude_after']) & (df_merged['longitude_before'] == df_merged['longitude_after'])
    df_merged.loc[start, ['inferred_agent_lat', 'inferred_agent_lon']] = df_merged.loc[start, ['latitude_after', 'longitude_after']].values
    df_merged.loc[end, ['inferred_agent_lat', 'inferred_agent_lon']] = df_merged.loc[end, ['latitude_before', 'longitude_before']].values
    df_merged.loc[start | end | unanchored, ['inferred_agent_speed_mps', 'inferred_agent_bearing']] = np.nan
    df_merged.loc[stationary, 'inferred_agent_speed_mps'] = 0
    df_merged['inferred_agent_bearing'] = df_merged.groupby(TARGET_SESSION_COL)['inferred_agent_bearing'].ffill()
    df_merged['inferred_agent_speed_mps'].fillna(0, inplace=True)
    df_merged['interpolation_quality'] = np.select(
        [unanchored, start, end, stationary],
        ['UNANCHORED', 'EXTRAPOLATED_START', 'EXTRAPOLATED_END', 'INTERPOLATED_STATIONARY'],
        default='INTERPOLATED'
    )
    print("  > Orphans, unanchored, and stationary events handled.")

    # --- 8. STAGE 5: PRESERVATION PROTOCOL Final Assembly ---
    print("\n--- STAGE 5: Assembling Final Asset via Index Join ---")
    # --- THIS IS THE MODIFIED LOGIC ---
    # Define all columns to be added, including the session ID used for processing
    cols_to_add = ['inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing', 'inferred_agent_speed_mps', 'interpolation_quality', TARGET_SESSION_COL]

    df_merged.drop_duplicates(subset=[TARGET_ID_COL], keep='first', inplace=True)
    df_merged.set_index(TARGET_ID_COL, inplace=True)
    print(f"  > Processed data de-duplicated and indexed on '{TARGET_ID_COL}'.")

    # Prepare the master dataframe by safely dropping the original session column before joining the new one
    df_targets_master_clean = df_targets_master.drop(columns=[TARGET_SESSION_COL], errors='ignore')

    # Join the results back to the original, complete, indexed master dataframe
    df_final = df_targets_master_clean.join(df_merged[cols_to_add])

    final_row_count = len(df_final)

    full_output_path = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_FILENAME)
    df_final.to_csv(full_output_path, index=True) # Save with the Obs_ID index

    print(f"\n✅ Mission Success. Definitive asset with {final_row_count} rows saved to:\n   > '{full_output_path}'")
    if initial_target_count == final_row_count:
        print(f"  > VALIDATION SUCCESS: All {initial_target_count} original rows have been preserved.")
    else:
        print(f"  > 🚨 VALIDATION FAILURE: Row count mismatch! Expected {initial_target_count}, got {final_row_count}.")

except Exception as e:
    import traceback; print(f"\n❌ An unrecoverable error occurred: {e}"); traceback.print_exc()

Authenticating & Mounting...
Mounted at /content/drive
✅ Setup complete.

--- STAGE 1: Data Ingestion & Session Annotation ---


/tmp/ipython-input-3314565787.py:56: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_targets_master.replace('', np.nan, inplace=True)


  > Master target dataframe loaded with 4776 rows.
  > Annotating anchors with ground-truth sessions...
  > Anchors successfully annotated.
  > Cleaned dataframe for processing contains 4766 valid rows.

--- STAGE 2: Definitive Session-Aware Merge ---
  > Explicit merge-rename-join protocol complete.

--- STAGE 3: Calculating Inferred State Vektors ---
  > Vektor calculations complete.

--- STAGE 4: Advanced Edge Case Handling ---


/tmp/ipython-input-3314565787.py:79: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_anchors[TARGET_SESSION_COL].fillna(method='bfill', inplace=True)
/tmp/ipython-input-3314565787.py:79: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_anchors[TARGET_SESSION_COL].fillna(method='bfill', inplace=True)
/tmp/ipython-input-3314565787.py:124: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignmen

  > Orphans, unanchored, and stationary events handled.

--- STAGE 5: Assembling Final Asset via Index Join ---
  > Processed data de-duplicated and indexed on 'Obs_ID'.

✅ Mission Success. Definitive asset with 4776 rows saved to:
   > '/content/drive/MyDrive/_Pienza/Assets/Database/03_analysis/raw_Offers_with_Inferred_Vektor_v5.4_PRESERVED.csv'
  > VALIDATION SUCCESS: All 4776 original rows have been preserved.


In [4]:
# ==============================================================================
# SCRIPT:         graph_analyzer_v1.0.ipynb
# OPERATION:      Operation: Network Genesis
# PURPOSE:        To construct and visualize a network graph from the Vektor
#                 trajectory data.
# ==============================================================================

# --- 1. SETUP ---
!pip install -q gspread networkx folium
import pandas as pd
import numpy as np
import os
import gspread
import networkx as nx
import folium
from google.colab import auth, drive
from google.auth import default

print("Authenticating & Mounting...")
auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)
drive.mount('/content/drive', force_remount=True)
print("✅ Setup complete.")


# --- 2. CONFIGURATION ---
# Use the GTS-4 sheet as the "ground truth" for the agent's path
ANCHORS_SPREADSHEET_NAME, ANCHORS_WORKSHEET_NAME = 'GTS-4', '250918'
ANCHOR_TS_COL, ANCHOR_LAT_COL, ANCHOR_LON_COL = 'realTimestamp', 'latitude', 'longitude'
SESSION_GAP_THRESHOLD_MINUTES = 30


# --- 3. UTILITY FUNCTIONS ---
def make_headers_unique(header_list):
    counts, unique_headers = {}, []
    for i, header in enumerate(header_list):
        header = str(header).strip() if str(header).strip() != "" else f"blank_{i}"
        if header not in counts: counts[header] = 1; unique_headers.append(header)
        else: counts[header] += 1; unique_headers.append(f"{header}_{counts[header]}")
    return unique_headers


# --- 4. STAGE 1: Load and Process Anchor Data ---
print("\n--- STAGE 1: Loading Ground-Truth Anchors ---")
anchor_sheet=gc.open(ANCHORS_SPREADSHEET_NAME).worksheet(ANCHORS_WORKSHEET_NAME)
df_anchors = pd.DataFrame(anchor_sheet.get_all_values()[1:], columns=make_headers_unique(anchor_sheet.get_all_values()[0]))
df_anchors.replace('', np.nan, inplace=True)
df_anchors['timestamp_utc']=pd.to_datetime(df_anchors[ANCHOR_TS_COL], errors='coerce')
df_anchors[ANCHOR_LAT_COL]=pd.to_numeric(df_anchors[ANCHOR_LAT_COL], errors='coerce')
df_anchors[ANCHOR_LON_COL]=pd.to_numeric(df_anchors[ANCHOR_LON_COL], errors='coerce')
df_anchors.dropna(subset=['timestamp_utc', ANCHOR_LAT_COL, ANCHOR_LON_COL], inplace=True)
df_anchors.sort_values(by='timestamp_utc', inplace=True, ignore_index=True)

print("  > Performing intelligent session detection...")
time_diff = df_anchors['timestamp_utc'].diff().dt.total_seconds() / 60
session_breaks = time_diff > SESSION_GAP_THRESHOLD_MINUTES
df_anchors['session_id'] = session_breaks.cumsum()
print(f"  > Detected {df_anchors['session_id'].nunique()} distinct work sessions.")


# --- 5. STAGE 2: Construct the Network Graph ---
print("\n--- STAGE 2: Building the NetworkX Graph ---")

# Use a DiGraph for directionality (A->B is different from B->A)
G = nx.DiGraph()

# Nodes are unique locations, identified by a rounded lat/lon tuple
# Rounding helps group very close pings into a single node
df_anchors['node_id'] = list(zip(round(df_anchors[ANCHOR_LAT_COL], 5), round(df_anchors[ANCHOR_LON_COL], 5)))

# Add nodes with their position attribute for plotting
for idx, row in df_anchors.iterrows():
    G.add_node(row['node_id'], pos=(row[ANCHOR_LAT_COL], row[ANCHOR_LON_COL]))

# Add edges by iterating through each session's anchor points
for session in df_anchors['session_id'].unique():
    session_df = df_anchors[df_anchors['session_id'] == session]
    # Create pairs of (current_point, next_point)
    for i in range(len(session_df) - 1):
        start_node = session_df.iloc[i]['node_id']
        end_node = session_df.iloc[i+1]['node_id']

        # Add an edge, increasing its weight if we've seen this path before
        if G.has_edge(start_node, end_node):
            G[start_node][end_node]['weight'] += 1
        else:
            G.add_edge(start_node, end_node, weight=1)

print(f"  > Graph construction complete.")
print(f"  > Total Nodes (Unique Locations): {G.number_of_nodes()}")
print(f"  > Total Edges (Travel Segments): {G.number_of_edges()}")


# --- 6. STAGE 3: Visualize the Graph on a Map with Folium ---
print("\n--- STAGE 3: Generating Geospatial Visualization with Folium ---")

# Calculate the center of the map
avg_lat = df_anchors[ANCHOR_LAT_COL].mean()
avg_lon = df_anchors[ANCHOR_LON_COL].mean()

# Create a base map
m = folium.Map(location=[avg_lat, avg_lon], zoom_start=13, tiles='CartoDB positron')

# Get node degrees to size the markers (a measure of importance/traffic)
degrees = dict(G.degree())
max_degree = max(degrees.values()) if degrees else 1

# Plot Edges (Travel Segments)
# Thicker lines for more frequently traveled routes
for start_node, end_node, data in G.edges(data=True):
    start_pos = G.nodes[start_node]['pos']
    end_pos = G.nodes[end_node]['pos']
    weight = data['weight']

    folium.PolyLine(
        locations=[start_pos, end_pos],
        weight=min(weight * 1.5, 10), # Scale line width by weight, with a max
        color='#FF5733',
        opacity=min(0.3 + (weight / 10), 0.9)
    ).add_to(m)

# Plot Nodes (Unique Locations)
# Bigger circles for more frequented locations
for node, data in G.nodes(data=True):
    pos = data['pos']
    degree = degrees.get(node, 1)

    folium.CircleMarker(
        location=pos,
        radius=3 + (degree / max_degree) * 10, # Scale radius by degree
        color='#0078FF',
        fill=True,
        fill_color='#0078FF',
        fill_opacity=0.6,
        popup=f"Location: {node}<br>Traffic (Degree): {degree}"
    ).add_to(m)


print("  > Map generation complete. The map object 'm' is ready for display.")

# To display the map in your Colab notebook, simply have 'm' as the last line in a cell
m

Output hidden; open in https://colab.research.google.com to view.

In [17]:
# ==============================================================================
# SCRIPT:         colab_vektor_engine_v5.4_indexed.ipynb
# OPERATION:      Operation: Vektor v5.4 (Corrected Indexing Protocol)
# PURPOSE:        The definitive asset. Corrects the ValueError by treating
#                 Obs_ID as a column throughout processing and only using it
#                 as an index during the final join.
# ==============================================================================

# --- 1. SETUP ---
!pip install -q gspread folium
import pandas as pd
import numpy as np
import os
import gspread
import folium
import time
from google.colab import auth, drive
from google.auth import default

print("Authenticating & Mounting...")
auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)
drive.mount('/content/drive', force_remount=True)
print("✅ Setup complete.")

# --- 2. CONFIGURATION ---
ANCHORS_SPREADSHEET_NAME, ANCHORS_WORKSHEET_NAME = 'GTS-4', '250918'
TARGETS_SPREADSHEET_NAME, TARGETS_WORKSHEET_NAME = 'raw_Offers', 'raw_requests_VVSpolished'
ANCHOR_TS_COL, ANCHOR_LAT_COL, ANCHOR_LON_COL = 'realTimestamp', 'latitude', 'longitude'
TARGET_TS_COL, TARGET_ID_COL = 'timestamp', 'Obs_ID'
TARGET_SESSION_COL = 'Session_ID'
OUTPUT_FOLDER_PATH = '/content/drive/MyDrive/_Pienza/Assets/Database/03_analysis'
OUTPUT_FILENAME = 'raw_Offers_with_Inferred_Vektor_v5.4_PRESERVED.csv'

# --- 3. BATTLE-TESTED UTILITY FUNCTIONS ---
def make_headers_unique(header_list):
    counts, unique_headers = {}, []
    for i, header in enumerate(header_list):
        header = str(header).strip() if str(header).strip() != "" else f"blank_{i}"
        if header not in counts:
            counts[header] = 1; unique_headers.append(header)
        else:
            counts[header] += 1; unique_headers.append(f"{header}_{counts[header]}")
    return unique_headers

def haversine_distance(lat1, lon1, lat2, lon2): R=6371000; phi1,phi2=np.radians(lat1),np.radians(lat2); dphi,dlambda=np.radians(lat2-lat1),np.radians(lon2-lon1); a=np.sin(dphi/2)**2+np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2; return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a))
def calculate_bearing(lat1, lon1, lat2, lon2): lat1,lon1,lat2,lon2=np.radians(lat1),np.radians(lon1),np.radians(lat2),np.radians(lon2); dLon=lon2-lon1; x=np.cos(lat2)*np.sin(dLon); y=np.cos(lat1)*np.sin(lat2)-np.sin(lat1)*np.cos(lat2)*np.cos(dLon); return (np.degrees(np.arctan2(x,y))+360)%360

try:
    # --- 4. STAGE 1: Data Ingestion & Ground-Truth Session Annotation ---
    print("\n--- STAGE 1: Data Ingestion & Session Annotation ---")

    target_sheet=gc.open(TARGETS_SPREADSHEET_NAME).worksheet(TARGETS_WORKSHEET_NAME)
    df_targets_master = pd.DataFrame(target_sheet.get_all_values()[1:], columns=make_headers_unique(target_sheet.get_all_values()[0]))
    df_targets_master.replace('', np.nan, inplace=True)
    initial_target_count = len(df_targets_master)
    print(f"  > Master target dataframe loaded with {initial_target_count} rows.")

    anchor_sheet=gc.open(ANCHORS_SPREADSHEET_NAME).worksheet(ANCHORS_WORKSHEET_NAME)
    df_anchors = pd.DataFrame(anchor_sheet.get_all_values()[1:], columns=make_headers_unique(anchor_sheet.get_all_values()[0]))
    df_anchors.replace('', np.nan, inplace=True)
    df_anchors['timestamp_utc']=pd.to_datetime(df_anchors[ANCHOR_TS_COL], errors='coerce')
    df_anchors.dropna(subset=['timestamp_utc'], inplace=True)
    df_anchors[ANCHOR_LAT_COL]=pd.to_numeric(df_anchors[ANCHOR_LAT_COL], errors='coerce')
    df_anchors[ANCHOR_LON_COL]=pd.to_numeric(df_anchors[ANCHOR_LON_COL], errors='coerce')
    df_anchors['timestamp_utc']=df_anchors['timestamp_utc'].dt.tz_localize('UTC')
    df_anchors.sort_values(by='timestamp_utc', inplace=True)

    print("  > Annotating anchors with ground-truth sessions...")
    session_map = df_targets_master.copy()
    session_map['timestamp_utc'] = pd.to_datetime(session_map[TARGET_TS_COL], format='%Y:%m:%d %H:%M:%S', errors='coerce').dt.tz_localize('UTC')
    session_map.dropna(subset=['timestamp_utc', TARGET_SESSION_COL], inplace=True)
    session_map = session_map[['timestamp_utc', TARGET_SESSION_COL]].sort_values('timestamp_utc')

    df_anchors = pd.merge_asof(df_anchors, session_map, on='timestamp_utc', direction='backward')
    df_anchors[TARGET_SESSION_COL].fillna(method='bfill', inplace=True)
    df_anchors.dropna(subset=[TARGET_SESSION_COL], inplace=True)
    print(f"  > Anchors successfully annotated.")

    df_targets_clean = df_targets_master.copy() # No indexing needed here
    df_targets_clean['timestamp_utc'] = pd.to_datetime(df_targets_clean[TARGET_TS_COL], format='%Y:%m:%d %H:%M:%S', errors='coerce')
    df_targets_clean.dropna(subset=['timestamp_utc', TARGET_ID_COL, TARGET_SESSION_COL], inplace=True)
    df_targets_clean['timestamp_utc'] = df_targets_clean['timestamp_utc'].dt.tz_localize('UTC')
    df_targets_clean.sort_values(by='timestamp_utc', inplace=True)
    print(f"  > Cleaned dataframe for processing contains {len(df_targets_clean)} valid rows.")

    # --- 5. STAGE 2: Session-Aware Merge ---
    print("\n--- STAGE 2: Definitive Session-Aware Merge ---")
    df_before = pd.merge_asof(df_targets_clean, df_anchors, on='timestamp_utc', by=TARGET_SESSION_COL, direction='backward')
    df_before.rename(columns={'latitude': 'latitude_before', 'longitude': 'longitude_before', 'realTimestamp':'timestamp_before'}, inplace=True)
    df_after = pd.merge_asof(df_targets_clean, df_anchors, on='timestamp_utc', by=TARGET_SESSION_COL, direction='forward')
    df_after.rename(columns={'latitude': 'latitude_after', 'longitude': 'longitude_after', 'realTimestamp':'timestamp_after'}, inplace=True)
    df_merged = df_before.join(df_after[['latitude_after', 'longitude_after', 'timestamp_after']])
    print("  > Explicit merge-rename-join protocol complete.")

    # --- 6. STAGE 3 & 7 (Combined): Vektor Calculation & Edge Cases ---
    print("\n--- STAGE 3 & 4: Calculating Vektors and Handling Edge Cases ---")
    df_merged['timestamp_before_utc'] = pd.to_datetime(df_merged['timestamp_before'], errors='coerce').dt.tz_localize('UTC')
    df_merged['timestamp_after_utc'] = pd.to_datetime(df_merged['timestamp_after'], errors='coerce').dt.tz_localize('UTC')
    time_total = (df_merged['timestamp_after_utc'] - df_merged['timestamp_before_utc']).dt.total_seconds()
    time_partial = (df_merged['timestamp_utc'] - df_merged['timestamp_before_utc']).dt.total_seconds()
    ratio = (time_partial / time_total).clip(0, 1).fillna(0)
    distance = haversine_distance(df_merged['latitude_before'], df_merged['longitude_before'], df_merged['latitude_after'], df_merged['longitude_after'])
    df_merged['inferred_agent_speed_mps'] = np.divide(distance, time_total)
    df_merged['inferred_agent_bearing'] = calculate_bearing(df_merged['latitude_before'], df_merged['longitude_before'], df_merged['latitude_after'], df_merged['longitude_after'])
    df_merged['inferred_agent_lat'] = df_merged['latitude_before'] + (df_merged['latitude_after'] - df_merged['latitude_before']) * ratio
    df_merged['inferred_agent_lon'] = df_merged['longitude_before'] + (df_merged['longitude_after'] - df_merged['longitude_before']) * ratio

    unanchored = df_merged['latitude_before'].isna() & df_merged['latitude_after'].isna()
    start = df_merged['latitude_before'].isna() & ~df_merged['latitude_after'].isna()
    end = ~df_merged['latitude_before'].isna() & df_merged['latitude_after'].isna()
    stationary = (df_merged['latitude_before'] == df_merged['latitude_after']) & (df_merged['longitude_before'] == df_merged['longitude_after'])

    df_merged.loc[start, ['inferred_agent_lat', 'inferred_agent_lon']] = df_merged.loc[start, ['latitude_after', 'longitude_after']].values
    df_merged.loc[end, ['inferred_agent_lat', 'inferred_agent_lon']] = df_merged.loc[end, ['latitude_before', 'longitude_before']].values
    df_merged.loc[start | end | unanchored, ['inferred_agent_speed_mps', 'inferred_agent_bearing']] = np.nan
    df_merged.loc[stationary, 'inferred_agent_speed_mps'] = 0
    df_merged['inferred_agent_bearing'] = df_merged.groupby(TARGET_SESSION_COL)['inferred_agent_bearing'].ffill()
    df_merged['inferred_agent_speed_mps'].fillna(0, inplace=True)
    df_merged['interpolation_quality'] = np.select(
        [unanchored, start, end, stationary],
        ['UNANCHORED', 'EXTRAPOLATED_START', 'EXTRAPOLATED_END', 'INTERPOLATED_STATIONARY'],
        default='INTERPOLATED'
    )
    print("  > Vektor logic complete.")

    # --- 8. STAGE 5: PRESERVATION PROTOCOL Final Assembly ---
    print("\n--- STAGE 5: Assembling Final Asset via Index Join ---")
    cols_to_add = ['inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing', 'inferred_agent_speed_mps', 'interpolation_quality']

    # --- JUST-IN-TIME INDEXING PROTOCOL ---
    df_targets_master.set_index(TARGET_ID_COL, inplace=True)
    df_merged.drop_duplicates(subset=[TARGET_ID_COL], keep='first', inplace=True)
    df_merged.set_index(TARGET_ID_COL, inplace=True)

    df_final = df_targets_master.join(df_merged[cols_to_add])
    final_row_count = len(df_final)

    full_output_path = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_FILENAME)
    df_final.to_csv(full_output_path, index=True) # index=True to save the Obs_ID

    print(f"\n✅ Mission Success. Definitive asset with {final_row_count} rows saved to:\n   > '{full_output_path}'")
    if initial_target_count == final_row_count:
        print(f"  > VALIDATION SUCCESS: All {initial_target_count} original rows have been preserved.")
    else:
        print(f"  > 🚨 VALIDATION FAILURE: Row count mismatch! Expected {initial_target_count}, got {final_row_count}.")

except Exception as e:
    import traceback; print(f"\n❌ An unrecoverable error occurred: {e}"); traceback.print_exc()

Authenticating & Mounting...
Mounted at /content/drive
✅ Setup complete.

--- STAGE 1: Data Ingestion & Session Annotation ---


/tmp/ipython-input-4048680580.py:54: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_targets_master.replace('', np.nan, inplace=True)


  > Master target dataframe loaded with 4776 rows.
  > Annotating anchors with ground-truth sessions...
  > Anchors successfully annotated.
  > Cleaned dataframe for processing contains 4766 valid rows.

--- STAGE 2: Definitive Session-Aware Merge ---
  > Explicit merge-rename-join protocol complete.

--- STAGE 3 & 4: Calculating Vektors and Handling Edge Cases ---
  > Vektor logic complete.

--- STAGE 5: Assembling Final Asset via Index Join ---


/tmp/ipython-input-4048680580.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_anchors[TARGET_SESSION_COL].fillna(method='bfill', inplace=True)
/tmp/ipython-input-4048680580.py:75: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_anchors[TARGET_SESSION_COL].fillna(method='bfill', inplace=True)
/tmp/ipython-input-4048680580.py:118: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignmen


✅ Mission Success. Definitive asset with 4776 rows saved to:
   > '/content/drive/MyDrive/_Pienza/Assets/Database/03_analysis/raw_Offers_with_Inferred_Vektor_v5.4_PRESERVED.csv'
  > VALIDATION SUCCESS: All 4776 original rows have been preserved.


In [18]:
# ==============================================================================
# SCRIPT:         Validation Stage 2: Visual Audit (v3.3 - Session-Aware)
# PURPOSE:        Generates a CORRECT visual audit map by creating a separate
#                 "ground-truth skeleton" for each individual session.
# ==============================================================================

print("\n--- Validation Stage 2: Visual Audit (Session-Aware) ---")

try:
    # These are the in-memory DataFrames from the initial ingestion.
    df_anchors_to_plot = df_anchors.copy()
    df_inferred_to_plot = df_final.copy()

    # Create a base map centered on Mexico City
    map_center = [19.4326, -99.1332]
    m = folium.Map(location=map_center, zoom_start=11, tiles="cartodbpositron")

    # --- The CRITICAL FIX: Group the skeleton by session ---
    print("Generating session-aware ground-truth skeleton (blue lines)...")
    # We group the anchors by the 'session_group' (date) we created during ingestion.
    session_paths = df_anchors_to_plot.groupby('session_group')

    # Iterate through each session and draw its own, separate line.
    for session_name, session_group in session_paths:
        points = session_group[['latitude', 'longitude']].values.tolist()
        folium.PolyLine(points, color='blue', weight=1.5, opacity=0.6).add_to(m)

    # --- Plot the Inferred Points (Red Dots) ---
    print("Overlaying inferred offer locations (red dots)...")
    # Plot all points now that we know they are within the bounding box.
    for _, row in df_inferred_to_plot.iterrows():
        if pd.notna(row['inferred_agent_lat']) and pd.notna(row['inferred_agent_lon']):
            # Color-code by quality
            if row['interpolation_quality'] == 'INTERPOLATED':
                dot_color = 'red'
            else: # EXTRAPOLATED
                dot_color = 'orange'

            folium.CircleMarker(
                location=[row['inferred_agent_lat'], row['inferred_agent_lon']],
                radius=2.5,
                color=dot_color,
                fill=True,
                fill_color=dot_color,
                fill_opacity=0.9,
                popup=f"Offer: {row['event_id']}<br>Quality: {row['interpolation_quality']}"
            ).add_to(m)

    # --- Save the map ---
    map_filename = 'trajectory_validation_map_DEFINITIVE.html'
    map_output_path = os.path.join(OUTPUT_FOLDER_PATH, map_filename)
    m.save(map_output_path)

    print(f"\n✅ VERIFICATION COMPLETE: The definitive, corrected map has been saved to your Google Drive.")
    print(f"   > '{map_output_path}'")
    print("   Please download and open this HTML file in your browser to visually inspect the results.")

except NameError:
    print("❌ ERROR: The DataFrames 'df_final' or 'df_anchors' were not found. Please ensure the main script cell ran successfully.")
except Exception as e:
    print(f"❌ An error occurred during map generation: {e}")


--- Validation Stage 2: Visual Audit (Session-Aware) ---
Generating session-aware ground-truth skeleton (blue lines)...
❌ An error occurred during map generation: 'session_group'
